# Session 4 — Weights, Systematics, Fitting, and Limits

In the final session we treat the bb + MET selection as a **full analysis**: we include event weights and simple systematic uncertainties, perform a binned likelihood fit of signal + backgrounds to data, assess goodness of fit, and derive an upper limit on the signal strength. This mirrors, in a simplified way, the statistical treatment used in the CMS bb + MET JHEP result.

**Learning objectives:**
- Understand event weights and systematic uncertainties in a bb + MET analysis
- Perform a binned likelihood fit of signal + backgrounds to data in MET bins
- Assess goodness of fit (χ², p-values, pulls)
- Compute an upper limit on signal strength for the bbMET signal

**Duration:** ~3 hours

**Input:** This session uses the processor output (e.g. `output_2017.pkl`) from Session 3. Run `python run_analysis.py` or `run_analysis.py --full` first if you do not have the pickle file.

![Placeholder: schematic limit plot (signal strength vs mediator mass or tanβ)](figures/session4_fig_limits_placeholder.png)

---
## 0. Load processor output

Load the per-dataset histograms and cutflows. Each key in `results` is a dataset name; each value is an accumulator with `met_sr`, `cutflow`, etc.

In [ ]:
# Ensure project root is on sys.path (for SWAN / any kernel)
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Load processor output (from run_analysis.py; saved to output/)
import os
def _load_results():
    for path in ["output/output_2017.pkl", "output/output_2017_full.pkl", "output_2017.pkl", "output_2017_full.pkl"]:
        if os.path.isfile(path):
            return pickle.load(open(path, "rb"))
    raise FileNotFoundError(
        "No processor output found. Run: python run_analysis.py (or run_analysis.py --full) first. "
        "Outputs are in output/output_2017.pkl or output/output_2017_full.pkl."
    )
results = _load_results()

print("Datasets:", list(results.keys())[:10], "..." if len(results) > 10 else "")
example = list(results.values())[0]
print("Accumulator keys:", list(example.keys()))
if "cos_theta_star_sr" in example:
    print("Main observable: cos_theta_star_sr")
elif "met_sr" in example:
    print("Main observable: met_sr")
if "met_sr" in example:
    print("met_sr axes:", example["met_sr"].axes[0].name)

---
## 1. Weights and systematics

The processor uses **genWeight** (sign only for MC) when filling histograms. In a full analysis we would add scale factors (e.g. pileup, lepton ID) and systematic variations.

**Systematic uncertainties** affect the predicted yields and shapes:
- **Normalisation:** e.g. ±20% per process (theory, luminosity).
- **Shape:** e.g. MET scale, b-tag efficiency, jet energy scale.

Here we introduce a simple **normalisation systematic** per background: we scale each histogram by a factor \((1 + \alpha \cdot \sigma)\), where \(\sigma\) is the relative uncertainty (e.g. 0.2) and \(\alpha\) is a nuisance parameter (fitted or fixed at ±1 for up/down).

In [ ]:
# Build nominal histogram per dataset. Main observable: cos_theta_star_sr (else met_sr).
from hist import Hist, axis

def get_met_sr_hist(results, dataset_name):
    """Return the met_sr histogram for a dataset."""
    if dataset_name not in results:
        return None
    return results[dataset_name]["met_sr"]

def get_main_hist(results, dataset_name):
    """Return cos_theta_star_sr (main observable) if present, else met_sr."""
    if dataset_name not in results:
        return None
    acc = results[dataset_name]
    return acc.get("cos_theta_star_sr") or acc["met_sr"]

# Example: list background and data dataset names (adjust to your results keys)
# Backgrounds: typically MC (ttbar, Zvv, Wjets, etc.); data: MET_Run2017* or Single*
all_datasets = list(results.keys())
data_datasets = [d for d in all_datasets if "Run2017" in d or "Single" in d or "MET" in d]
bkg_datasets = [d for d in all_datasets if d not in data_datasets]
print("Data (example):", data_datasets[:3])
print("Backgrounds (example):", bkg_datasets[:5])

---
## 2. Binned fit

We fit a **binned likelihood**: in each MET bin \(i\), the observed count \(n_i\) is Poisson with mean \(\mu \cdot s_i + \sum_b b_{i,b}\), where \(s_i\) is the signal prediction, \(b_{i,b}\) the background \(b\) in bin \(i\), and \(\mu\) is the signal strength (1 = SM signal, 0 = background only).

**Simplified:** Assume only backgrounds (no signal histogram yet). Fit the sum of background histograms to the data histogram by scaling backgrounds with normalisation factors (or fix them and compute \(\chi^2\)). If you have a signal histogram, add \(\mu \cdot s\) and fit \(\mu\).

In [ ]:
from scipy.optimize import minimize
from scipy.stats import poisson

# Get bin edges and bin counts from histograms (hist library)
def hist_values_edges(h):
    """Return (values, bin_edges) for a 1D hist."""
    vals = h.values()
    ax = h.axes[0]
    edges = ax.edges
    return np.asarray(vals), np.asarray(edges)

# Sum background histograms (same binning)
def sum_histograms(results, dataset_list):
    h0 = None
    for d in dataset_list:
        h = get_met_sr_hist(results, d)
        if h is None:
            continue
        if h0 is None:
            h0 = h.copy()
        else:
            h0 = h0 + h
    return h0

# Example: data = sum of data datasets; background = sum of bkg_datasets
# Then define negative log-likelihood (Poisson) and minimize w.r.t. a global scale.
# Your code: build data_hist, bkg_hist, then fit (e.g. one scale factor for all bkg).

---
## 3. Goodness of fit

After the fit, check how well the model describes the data:
- **\(\chi^2\):** \(\chi^2 = \sum_i (n_i - \lambda_i)^2 / \lambda_i\) (or use variance \(\lambda_i\) for Poisson).
- **Degrees of freedom:** number of bins minus number of fitted parameters.
- **p-value:** \(P(\chi^2 \geq \chi^2_{\text{obs}})\) from the \(\chi^2\) distribution.
- **Pulls:** \((n_i - \lambda_i) / \sigma_i\) per bin to spot which bins disagree.

In [ ]:
from scipy.stats import chi2

# Example: after you have data counts and fitted model (expected) per bin
# observed = ...  # shape (n_bins,)
# expected = ... # shape (n_bins,)
# chi2_val = np.sum((observed - expected)**2 / np.where(expected > 0, expected, 1))
# ndof = len(observed) - n_fitted_params
# pvalue = 1 - chi2.cdf(chi2_val, ndof)
# print(f"chi2 = {chi2_val:.2f}, ndof = {ndof}, p-value = {pvalue:.4f}")
# Pulls: (observed - expected) / np.sqrt(expected)
# Your code: compute chi2, ndof, p-value, and optionally plot pulls.

---
## 4. Limit calculation

An **upper limit** on the signal strength \(\mu\) at e.g. 95% CL says: we exclude \(\mu > \mu_{\text{limit}}\) at 95% confidence. Common methods:
- **Asymptotic formula:** Use the profile likelihood ratio and the asymptotic distribution (e.g. \(2\Delta\ln L\) is chi-squared distributed).
- **pyhf:** The [pyhf](https://pyhf.readthedocs.io/) library implements histogram fits and asymptotic limits (and toys if needed).

**Simplified exercise:** For a single bin (e.g. total SR yield), the 95% CL upper limit on the number of signal events can be computed with the one-sided Poisson formula (e.g. \(n_{\text{obs}}\) observed, \(b\) expected background → upper limit on \(s\)). Or use scipy to find \(\mu\) such that the profile likelihood (or a simple likelihood) gives the desired CL.

In [ ]:
# Example: 95% CL upper limit on signal strength mu (single bin or binned)
# Option A: Use asymptotic approximation (e.g. 1.64 sigma one-sided -> approximate mu_limit)
# Option B: pyhf - build a workspace and run pyhf.infer.hypotest(1, [...], ...)
# Your code: implement a simple limit (e.g. total yield in SR, Poisson(n_obs | b + s), find s_95).

---
## 5. Summary

- **Weights:** MC event weights (e.g. genWeight) are used when filling histograms; systematics are encoded as normalisation or shape variations.
- **Fit:** Binned Poisson likelihood fit of signal + backgrounds to data; minimisation gives best-fit parameters (e.g. signal strength \(\mu\)).
- **Goodness of fit:** \(\chi^2\), p-value, and pulls quantify how well the model describes the data.
- **Limit:** Upper limit on signal (e.g. 95% CL) via asymptotic formula or pyhf.

**Further reading:** pyhf documentation, TRExFitter, Combine (CMS).

### Exercises

1. Add a second systematic (e.g. shape: scale MET by 1.02 for "up") and re-fit.
2. Compute the limit for a different signal hypothesis (e.g. different cross-section).
3. (If time) Compare asymptotic limit with a toy Monte Carlo limit.